# 02: Feature Engineering

**Author:** Karan Homayounfar (25065219), UWE Bristol

This notebook builds the scikit-learn preprocessing pipeline and outputs clean train/test arrays for modelling.  
Scores primarily on **C2 Application/Design (30%)**.

**Key design decisions:**
- The train/test split is temporal (2020-2024 train, 2025-2026 test), not random. This reflects actual deployment.
- Potential efficiency columns are excluded to prevent target leakage.
- A `y_train_years` array is saved alongside X/y so that model notebooks can use expanding-window temporal CV.

**Outputs:** `X_train.npy`, `X_test.npy`, `y_train.npy`, `y_test.npy`, `y_train_years.npy`, `feature_names.pkl`, `preprocessor.pkl`

In [1]:
import pandas as pd
import numpy as np
import pickle
import sys
import os
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, '../src')

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer

TRAIN_SAMPLE = '../data/processed/epc_train_sample_200k.parquet'
TEST_SAMPLE  = '../data/processed/epc_test_sample_50k.parquet'
OUT_DIR      = '../data/processed'

os.makedirs(OUT_DIR, exist_ok=True)

## 1. Load data

In [ ]:
train = pd.read_parquet(TRAIN_SAMPLE)
test  = pd.read_parquet(TEST_SAMPLE)

# Parse lodgement year for temporal CV
train['YEAR'] = pd.to_datetime(train['LODGEMENT_DATE'], errors='coerce').dt.year
test['YEAR']  = pd.to_datetime(test['LODGEMENT_DATE'],  errors='coerce').dt.year

# Drop rows with missing year
train = train.dropna(subset=['YEAR']).copy()
test  = test.dropna(subset=['YEAR']).copy()
train['YEAR'] = train['YEAR'].astype(int)
test['YEAR']  = test['YEAR'].astype(int)

print(f'Train: {train.shape}')
print('Train year distribution:')
print(train['YEAR'].value_counts().sort_index())
print(f'\nTest: {test.shape}')
print('Test year distribution:')
print(test['YEAR'].value_counts().sort_index())

# Data quality: drop physically impossible negative values in energy/emissions/cost fields.
# These are sentinel or entry errors in the source EPC register (identified in 01_EDA), not
# real measurements, so the whole record is removed. Same principle as the efficiency
# range filter applied in data preparation. Affects ~0.1% of rows.
NON_NEGATIVE_COLS = [
    'CURRENT_ENERGY_EFFICIENCY', 'TOTAL_FLOOR_AREA',
    'NUMBER_HABITABLE_ROOMS', 'NUMBER_HEATED_ROOMS',
    'CO2_EMISS_CURR_PER_FLOOR_AREA', 'CO2_EMISSIONS_CURRENT',
    'ENERGY_CONSUMPTION_CURRENT', 'HEATING_COST_CURRENT',
    'HOT_WATER_COST_CURRENT', 'LIGHTING_COST_CURRENT',
    'EXTENSION_COUNT', 'FIXED_LIGHTING_OUTLETS_COUNT',
    'MULTI_GLAZE_PROPORTION', 'LOW_ENERGY_LIGHTING',
]
for _df_name, _df in [('train', train), ('test', test)]:
    _present = [c for c in NON_NEGATIVE_COLS if c in _df.columns]
    _bad = (_df[_present] < 0).any(axis=1)
    print(f'{_df_name}: dropping {int(_bad.sum())} rows with negative physical values ({_bad.mean()*100:.3f}%)')

_p = [c for c in NON_NEGATIVE_COLS if c in train.columns]
train = train[~(train[_p] < 0).any(axis=1)].copy()  # NaN kept (imputed later)
_p = [c for c in NON_NEGATIVE_COLS if c in test.columns]
test = test[~(test[_p] < 0).any(axis=1)].copy()
print(f'After negative-value removal -> Train: {train.shape}, Test: {test.shape}')

## 2. Extract wall type

In [3]:
def extract_wall_type(df):
    df = df.copy()
    if 'WALLS_DESCRIPTION' in df.columns:
        desc = df['WALLS_DESCRIPTION'].str.lower().fillna('')
        df['WALL_TYPE'] = 'other'
        df.loc[desc.str.contains('cavity'), 'WALL_TYPE'] = 'cavity'
        df.loc[desc.str.contains('solid'),  'WALL_TYPE'] = 'solid'
    else:
        df['WALL_TYPE'] = 'unknown'
    return df

train = extract_wall_type(train)
test  = extract_wall_type(test)
print('WALL_TYPE (train):', train['WALL_TYPE'].value_counts().to_dict())

WALL_TYPE (train): {'cavity': 107576, 'other': 52130, 'solid': 40294}


## 3. Define feature groups

**Leakage exclusions:** `POTENTIAL_ENERGY_EFFICIENCY`, `POTENTIAL_ENERGY_RATING`, `EFFICIENCY_GAP`, `RETROFIT_POTENTIAL`.  
These are derived from or define the target. Including them would give trivially perfect performance.

In [4]:
NUMERIC_COLS = [
    'CURRENT_ENERGY_EFFICIENCY', 'TOTAL_FLOOR_AREA',
    'NUMBER_HABITABLE_ROOMS', 'NUMBER_HEATED_ROOMS',
    'CO2_EMISS_CURR_PER_FLOOR_AREA', 'CO2_EMISSIONS_CURRENT',
    'ENERGY_CONSUMPTION_CURRENT', 'HEATING_COST_CURRENT',
    'HOT_WATER_COST_CURRENT', 'LIGHTING_COST_CURRENT',
    'EXTENSION_COUNT', 'FIXED_LIGHTING_OUTLETS_COUNT',
    'MULTI_GLAZE_PROPORTION', 'LOW_ENERGY_LIGHTING',
]

ENERGY_RATING_COLS  = ['CURRENT_ENERGY_RATING']
ENERGY_RATING_ORDER = [['G', 'F', 'E', 'D', 'C', 'B', 'A']]  # G=0 -> A=6

EFF_RATING_COLS = [
    'WALLS_ENERGY_EFF', 'ROOF_ENERGY_EFF', 'FLOOR_ENERGY_EFF',
    'WINDOWS_ENERGY_EFF', 'MAINHEAT_ENERGY_EFF', 'HOT_WATER_ENERGY_EFF',
    'LIGHTING_ENERGY_EFF', 'MAINHEATC_ENERGY_EFF',
]
EFF_RATING_VALS = [['N/A', 'Very Poor', 'Poor', 'Average', 'Good', 'Very Good']] * len(EFF_RATING_COLS)

NOMINAL_COLS = [
    'PROPERTY_TYPE', 'BUILT_FORM', 'TENURE',
    'MAINS_GAS_FLAG', 'TRANSACTION_TYPE', 'WALL_TYPE',
]

def cols_present(cols, df):
    return [c for c in cols if c in df.columns]

numeric_cols    = cols_present(NUMERIC_COLS, train)
energy_r_cols   = cols_present(ENERGY_RATING_COLS, train)
eff_rating_cols = cols_present(EFF_RATING_COLS, train)
nominal_cols    = cols_present(NOMINAL_COLS, train)

print(f'Numeric: {len(numeric_cols)}, EnergyRating: {len(energy_r_cols)}, EffRating: {len(eff_rating_cols)}, Nominal: {len(nominal_cols)}')

Numeric: 14, EnergyRating: 1, EffRating: 8, Nominal: 6


## 4. Build sklearn Pipeline (fit on train only)

In [5]:
transformers = []

if numeric_cols:
    transformers.append(('num', Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('scale', StandardScaler()),
    ]), numeric_cols))

if energy_r_cols:
    transformers.append(('energy_rating', Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent')),
        ('ord', OrdinalEncoder(
            categories=ENERGY_RATING_ORDER,
            handle_unknown='use_encoded_value', unknown_value=-1
        )),
    ]), energy_r_cols))

if eff_rating_cols:
    eff_order = EFF_RATING_VALS[:len(eff_rating_cols)]
    transformers.append(('eff_rating', Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent')),
        ('ord', OrdinalEncoder(
            categories=eff_order,
            handle_unknown='use_encoded_value', unknown_value=-1
        )),
    ]), eff_rating_cols))

if nominal_cols:
    transformers.append(('nom', Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(
            handle_unknown='ignore', sparse_output=False, drop='first'
        )),
    ]), nominal_cols))

preprocessor = ColumnTransformer(transformers=transformers, remainder='drop')
all_feat_cols = numeric_cols + energy_r_cols + eff_rating_cols + nominal_cols

y_train = train['RETROFIT_POTENTIAL'].values
y_test  = test['RETROFIT_POTENTIAL'].values
y_train_years = train['YEAR'].values  # for temporal CV in model notebooks

X_train = preprocessor.fit_transform(train[all_feat_cols])   # fit on train only
X_test  = preprocessor.transform(test[all_feat_cols])

print(f'X_train: {X_train.shape}, y_train positive: {y_train.mean():.3f}')
print(f'X_test:  {X_test.shape},  y_test positive:  {y_test.mean():.3f}')
assert not np.isnan(X_train).any(), 'NaN in X_train'
assert not np.isnan(X_test).any(),  'NaN in X_test'
print('No NaN values. Pipeline check passed.')

X_train: (200000, 51), y_train positive: 0.217
X_test:  (50000, 51),  y_test positive:  0.108
No NaN values. Pipeline check passed.


## 5. Extract feature names

In [6]:
feature_names = []
feature_names.extend(numeric_cols)
feature_names.extend(energy_r_cols)
feature_names.extend(eff_rating_cols)
if nominal_cols:
    ohe_names = preprocessor.named_transformers_['nom']['ohe'].get_feature_names_out(nominal_cols).tolist()
    feature_names.extend(ohe_names)

print(f'Total features after encoding: {len(feature_names)}')

Total features after encoding: 51


## 6. Save all outputs

In [7]:
np.save(f'{OUT_DIR}/X_train.npy',       X_train)
np.save(f'{OUT_DIR}/X_test.npy',        X_test)
np.save(f'{OUT_DIR}/y_train.npy',       y_train)
np.save(f'{OUT_DIR}/y_test.npy',        y_test)
np.save(f'{OUT_DIR}/y_train_years.npy', y_train_years)

with open(f'{OUT_DIR}/feature_names.pkl', 'wb') as f: pickle.dump(feature_names, f)
with open(f'{OUT_DIR}/preprocessor.pkl',  'wb') as f: pickle.dump(preprocessor,  f)

print('Saved:')
for name, arr in [('X_train', X_train), ('X_test', X_test),
                  ('y_train', y_train), ('y_test', y_test),
                  ('y_train_years', y_train_years)]:
    print(f'  {name}.npy  shape={arr.shape}')
print(f'  feature_names.pkl  ({len(feature_names)} features)')
print(f'  preprocessor.pkl')

Saved:
  X_train.npy  shape=(200000, 51)
  X_test.npy  shape=(50000, 51)
  y_train.npy  shape=(200000,)
  y_test.npy  shape=(50000,)
  y_train_years.npy  shape=(200000,)
  feature_names.pkl  (51 features)
  preprocessor.pkl


## Feature engineering decisions

| Decision | Chosen | Rejected | Reason |
|---|---|---|---|
| Encode CURRENT_ENERGY_RATING | Ordinal (G=0 to A=6) | OHE | Preserves natural order |
| Encode component efficiency ratings | Ordinal (N/A=0 to Very Good=5) | OHE | Six ordered categories |
| PROPERTY_TYPE, BUILT_FORM | OHE drop=first | Label encoding | No ordinal relationship |
| Missing numerics | Median | Mean or drop | Robust to floor-area outliers |
| Missing categoricals | Mode | Drop | Consistent imputation strategy |
| Numeric scaling | StandardScaler | MinMaxScaler | More robust to outliers |
| WALLS_DESCRIPTION | Extract cavity/solid/other | Keep raw text | Tree and linear models cannot use raw text |
| POTENTIAL_ENERGY_EFFICIENCY | Excluded | Included | Direct leakage into target |
| CURRENT_ENERGY_EFFICIENCY | Included | Excluded | Observable feature, no leakage |
| Train/test split | Temporal 2020-2024 / 2025-2026 | Random | Prevents temporal leakage |
| Outer CV | Expanding-window by year | StratifiedKFold | Respects time structure of data |
| Negative physical values | Drop record | Clip to 0 / impute | Sentinel/entry errors (~0.1% of rows); dropping is consistent with the efficiency range filter and avoids inventing values |
